# SpikeDistill-RUL: reproducible project workflow

Run this notebook from the repository root (`D:/Ml`). It records the code pipeline for C-MAPSS FD001–FD004, from preprocessing through validation, teacher training, student training, ablations, explainability, and final refit/evaluation. **Jetson/edge deployment is intentionally out of scope for this notebook.**

Long training cells are deliberately explicit: run each stage when needed and inspect its logs before moving to the next. The official test split is reserved for the final evaluation stage; this project already has an earlier exploratory test evaluation in `results/official_test_evaluation`, so report the final test numbers with that limitation stated.


In [ ]:
from pathlib import Path
import subprocess, sys, json

ROOT = Path.cwd()
assert (ROOT / 'preprocess_cmapss.py').exists(), f'Open notebook from D:/Ml; current folder: {ROOT}'
PYTHON = sys.executable
# Turn a stage on to execute it; leave expensive stages off to inspect saved outputs.
RUN_STAGES = {
    'preprocess': False, 'baselines': False, 'sequence_baselines': False,
    'teacher': False, 'snn': False, 'tuning': False, 'refinement': False,
    'fd003_refinement': False, 'fd003_epoch_tune': False, 'ablations': False, 'xai': False, 'final_refit': False,
}

def run(script, *args):
    cmd = [PYTHON, str(ROOT / script), *map(str, args)]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)

def run_stage(name, script, *args):
    if RUN_STAGES[name]:
        run(script, *args)
    else:
        print(f'Skipping {name}; set RUN_STAGES[{name!r}] = True to run it.')

print('Repository:', ROOT)
print('Python:', sys.version)
print('CUDA available:', __import__('torch').cuda.is_available())


## 1. Prepare data and capped RUL windows

Preprocessing creates engine-wise train/validation splits, fits its feature transform on training data, makes 30-cycle windows, caps RUL at 125, and exports official test windows. Run only if you intend to regenerate the processed files.

In [ ]:
run_stage('preprocess', 'preprocess_cmapss.py')
processed = ROOT / 'CMAPSSData' / 'processed'
print('Final window files:', sorted(p.name for p in processed.glob('final_windows_FD*.npz')))


## 2. Tabular and sequence baselines

The first script trains mean, Ridge, Random Forest, and DeepNN baselines. The sequence script trains RNN, LSTM, GRU, BiLSTM, and CNN–LSTM baselines.

In [ ]:
run_stage('baselines', 'train_baselines.py')
run_stage('sequence_baselines', 'train_sequence_baselines.py')


## 3. Attention-BiLSTM teacher and recurrent SNN

Train the teacher first so its predictions/representations are available for distillation. Then train the SNN stages. These commands can take a long time on CPU.

In [ ]:
run_stage('teacher', 'train_bilstm_teacher.py')
run_stage('snn', 'train_spike_models.py')


## 4. Validation tuning and model refinement

The project scripts below search/refine SNN settings using train/validation data. Their saved histories and selection summaries are the source for the locked refit recipes. Run these only when reproducing the model-selection process from scratch.

In [ ]:
run_stage('tuning', 'tune_spikedistill.py')
run_stage('refinement', 'refine_spikedistill.py')
run_stage('fd003_refinement', 'refine_fd003.py')
run_stage('fd003_epoch_tune', 'tune_fd003_best_epoch.py')


## 5. Ablations and explainability

Ablations remove one distillation/loss component at a time across three seeds. Explainability exports integrated gradients, sensor permutation importance, temporal importance, attention summaries, and SNN spike/timestep attribution.

In [ ]:
run_stage('ablations', 'run_ablation_study.py')
run_stage('xai', 'explain_spikedistill.py', '--samples', 2, '--permutations', 4)


## 6. Final full-train refit and official test evaluation

The locked recipe is chosen from validation experiments, then teacher and SNN models are refit on all official training engines. Only after checkpoints are saved does the script load official test arrays and produce metrics/predictions. Do not tune against these test metrics.

In [ ]:
run_stage('final_refit', 'final_refit_eval.py')


## 7. Review existing outputs

This cell reads summaries if they exist; it does not start training or load the raw official test data.

In [ ]:
def show_csv(path):
    import pandas as pd
    p = ROOT / path
    if p.exists():
        display(pd.read_csv(p).head(20))
    else:
        print('Not available yet:', p)

show_csv('results/spike_models/ablations/seed_metrics.csv')
show_csv('results/final_refit_evaluation_aligned/metrics.csv')
